# MVP – Análisis de sentimiento (Negativo / Neutro / Positivo)
Pipeline exportable en **un solo** `.joblib` (TF‑IDF + Logistic Regression)

**Objetivo:** entrenar un clasificador ternario y exportarlo como `artifacts/sentiment_pipeline.joblib` para que un `predict.py` pueda hacer `pipeline.predict_proba([texto])` y usar `argmax`.

⚠️ Nota importante sobre reproducibilidad y warnings: este notebook asume **scikit-learn==1.1.3** y **joblib==1.2.0**.


## 0) Setup
### Dependencias recomendadas
Crea un entorno y fija versiones para evitar warnings al cargar el `.joblib`:

```bash
python -m venv .venv
source .venv/bin/activate  # Linux/Mac
# .venv\Scripts\activate  # Windows
pip install -U pip
pip install scikit-learn==1.1.3 joblib==1.2.0 pandas numpy matplotlib
```


In [ ]:
# Verifica versiones (deben coincidir con lo recomendado)
import sklearn, joblib, pandas as pd, numpy as np
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)


## 1) Carga del dataset
Se espera un CSV llamado `dataset_sentimientos_ecom (1).csv` con columnas:
- `texto` (string)
- `sentimiento` ∈ {`negativo`, `neutro`, `positivo`}


In [ ]:
from pathlib import Path

DATA_PATH = Path("dataset_sentimientos_ecom (1).csv")  # ajusta si tu archivo tiene otro nombre/ruta
assert DATA_PATH.exists(), f"No encuentro el CSV en: {DATA_PATH.resolve()}"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 2) ETL / Limpieza
Incluye:
- nulos
- duplicados
- normalización del label `sentimiento`


In [ ]:
# Normaliza nombres de columnas por si vienen con espacios/variantes
df.columns = [c.strip().lower() for c in df.columns]

required_cols = {"texto", "sentimiento"}
missing = required_cols - set(df.columns)
assert not missing, f"Faltan columnas requeridas: {missing}. Columnas actuales: {df.columns.tolist()}"

# Limpieza básica
df["texto"] = df["texto"].astype(str).fillna("").str.strip()
df["sentimiento"] = df["sentimiento"].astype(str).fillna("").str.strip().str.lower()

# Filtra labels válidos
valid = {"negativo", "neutro", "positivo"}
df = df[df["sentimiento"].isin(valid)].copy()

# Quita textos vacíos
df = df[df["texto"].str.len() > 0].copy()

# Drop duplicates (mismo texto y misma etiqueta)
before = len(df)
df = df.drop_duplicates(subset=["texto", "sentimiento"]).copy()
after = len(df)

print("Filas tras limpieza:", len(df))
print("Duplicados removidos:", before - after)

df.sample(5, random_state=42)


## 3) EDA mínimo
- Distribución de clases
- Ejemplos
- Longitud de texto (caracteres y palabras)


In [ ]:
import matplotlib.pyplot as plt

# Distribución de clases
class_counts = df["sentimiento"].value_counts()
display(class_counts)

plt.figure()
class_counts.plot(kind="bar")
plt.title("Distribución de clases")
plt.xlabel("sentimiento")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Longitud de texto
df["len_chars"] = df["texto"].str.len()
df["len_words"] = df["texto"].str.split().apply(len)

display(df[["len_chars","len_words"]].describe())

plt.figure()
plt.hist(df["len_words"], bins=30)
plt.title("Histograma: longitud (palabras)")
plt.xlabel("n_words")
plt.ylabel("count")
plt.show()

# Ejemplos por clase
for label in ["negativo", "neutro", "positivo"]:
    sample = df[df["sentimiento"] == label].sample(3, random_state=42)["texto"].tolist()
    print("\n---", label.upper(), "---")
    for i, t in enumerate(sample, 1):
        print(f"{i}. {t[:200]}{'...' if len(t) > 200 else ''}")


## 4) Función de limpieza usada por el modelo
⚠️ Importante para exportar: **NO** definas `limpiar_texto()` en la celda y ya.
Para que el `.joblib` se cargue en `predict.py` sin errores, la función debe vivir en un **módulo importable**.
Este notebook genera `text_cleaning.py` y luego importa `limpiar_texto` desde ahí.


In [ ]:
from pathlib import Path

cleaning_module = Path("text_cleaning.py")
cleaning_module.write_text(
"""import re

_URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
_MENTION_RE = re.compile(r"@\w+", flags=re.UNICODE)
# Conserva letras (incluyendo acentos), números y espacios. Reemplaza lo demás por espacio.
# Nota: después de lower(), incluimos explícitamente caracteres comunes en español.
_NON_ALNUM_ES_RE = re.compile(r"[^0-9a-záéíóúüñ\s]", flags=re.IGNORECASE)

def limpiar_texto(texto: str) -> str:
    """Limpieza mínima para NLP (MVP):
    - minúsculas
    - remover URLs y menciones
    - remover caracteres especiales (conservando acentos)
    - normalizar espacios
    """
    if texto is None:
        return ""
    texto = str(texto).lower()
    texto = _URL_RE.sub(" ", texto)
    texto = _MENTION_RE.sub(" ", texto)
    texto = _NON_ALNUM_ES_RE.sub(" ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto
""",
encoding="utf-8")

print("Archivo creado:", cleaning_module.resolve())


In [ ]:
from text_cleaning import limpiar_texto

# sanity check
tests = [
    "Excelente!! Entrega rápida. 10/10 😄 https://tienda.com",
    "@soporte no me resolvieron nada... pésimo servicio!!!",
    "Está bien, sin más."
]
for t in tests:
    print("IN :", t)
    print("OUT:", limpiar_texto(t))
    print("---")


## 5) Train/Test split (estratificado)

In [ ]:
from sklearn.model_selection import train_test_split

X = df["texto"].values
y = df["sentimiento"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", len(X_train), "Test:", len(X_test))


## 6) Entrenamiento – Pipeline (TF‑IDF + Logistic Regression)
Decisiones:
- **TF‑IDF**: baseline fuerte para texto corto (reseñas/comentarios), rápido y robusto.
- **LogisticRegression**: buen rendimiento lineal con TF‑IDF, `predict_proba` nativo.
- **macro‑F1**: balancea performance entre clases en clasificación ternaria, incluso si hay desbalance.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        preprocessor=limpiar_texto,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=50000,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        solver="saga",
        penalty="l2",
        C=4.0,
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1,
        multi_class="multinomial",
        random_state=42
    ))
])

pipeline


In [ ]:
# Entrena
pipeline.fit(X_train, y_train)

print("Clases aprendidas:", pipeline.classes_)


## 7) Evaluación
Incluye:
- `classification_report`
- matriz de confusión


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred, labels=pipeline.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipeline.classes_)
plt.figure()
disp.plot()
plt.title("Matriz de confusión")
plt.show()


## 8) (Opcional recomendado) Validación cruzada (StratifiedKFold=5) con F1 macro

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipeline,
    X_train, y_train,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

print("F1_macro CV (train) mean:", scores.mean())
print("F1_macro CV (train) std :", scores.std())
print("Scores:", scores)


## 9) Export del modelo (Pipeline completo en 1 solo .joblib)
Esto crea `artifacts/sentiment_pipeline.joblib`.

✅ Debe soportar:
- `pipeline.predict_proba([texto])`
- `pipeline.classes_`


In [ ]:
import os, joblib
from pathlib import Path

ART_DIR = Path("artifacts")
ART_DIR.mkdir(exist_ok=True)

MODEL_PATH = ART_DIR / "sentiment_pipeline.joblib"
joblib.dump(pipeline, MODEL_PATH)

print("Exportado a:", MODEL_PATH.resolve())
print("Tamaño (bytes):", MODEL_PATH.stat().st_size)


## 10) Prueba final de inferencia (5 ejemplos)
Muestra clase + probabilidad de la clase elegida + vector de probabilidades por clase.


In [ ]:
import time
import numpy as np

ejemplos = [
    "Excelente servicio, llegó rapidísimo y bien empacado.",
    "Me encantó la calidad, compraré otra vez.",
    "Está bien, cumple, pero no es la gran cosa.",
    "Pésimo, llegó roto y nadie responde.",
    "Muy malo, no era lo que decía la publicación."
]

for t in ejemplos:
    start = time.perf_counter()
    probs = pipeline.predict_proba([t])[0]
    pred_idx = int(np.argmax(probs))
    pred_label = pipeline.classes_[pred_idx]
    latency_ms = (time.perf_counter() - start) * 1000

    print("\nTexto:", t)
    print("Predicción:", pred_label)
    print("Probabilidad:", float(probs[pred_idx]))
    print("Probs por clase:", {cls: float(p) for cls, p in zip(pipeline.classes_, probs)})
    print("Latency (ms):", round(latency_ms, 3))


## 11) Snippet para Backend (predict.py standalone)
Este script carga el `.joblib`, calcula `predict_proba`, usa `argmax` y devuelve JSON.

Asegúrate de que `text_cleaning.py` esté en el mismo directorio o disponible en `PYTHONPATH`.


In [ ]:
# (Opcional) Genera el archivo predict.py desde el notebook
from pathlib import Path

Path("predict.py").write_text(
"""import argparse
import json
import time
import numpy as np
import joblib

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default="artifacts/sentiment_pipeline.joblib")
    parser.add_argument("--text", required=True)
    parser.add_argument("--pretty", action="store_true")
    args = parser.parse_args()

    t0 = time.perf_counter()
    pipeline = joblib.load(args.model)

    probs = pipeline.predict_proba([args.text])[0]
    idx = int(np.argmax(probs))
    pred = pipeline.classes_[idx]
    latency_ms = (time.perf_counter() - t0) * 1000.0

    out = {
        "prediction": str(pred),
        "probability": float(probs[idx]),
        "probs": {str(c): float(p) for c, p in zip(pipeline.classes_, probs)},
        "latency_ms": round(latency_ms, 3)
    }

    if args.pretty:
        print(json.dumps(out, ensure_ascii=False, indent=2))
    else:
        print(json.dumps(out, ensure_ascii=False))

if __name__ == "__main__":
    main()
""",
encoding="utf-8")
print("predict.py creado.")
